In [ ]:
# =========================================
# 1. DISTRIBUTIONAL ANALYSIS
# =========================================

# Accessing the Dataset in R
install.packages(c("modeldata", "fitdistrplus"))
library(modeldata)
library(fitdistrplus)
library(ggplot2)

data(ames)

# Identify and extract key numeric variables
SalePrice <- as.numeric(ames$Sale_Price)
GrLivArea <- as.numeric(ames$Gr_Liv_Area)

# Remove missing values
SalePrice <- SalePrice[!is.na(SalePrice)]
GrLivArea <- GrLivArea[!is.na(GrLivArea)]

# =========================================
# Exploratory distribution analysis (WIDE)
# =========================================

options(repr.plot.width = 14, repr.plot.height = 6)
par(mfrow = c(1,2))

hist(SalePrice, breaks = 40, col = "lightblue",
     main = "Distribution of Sale Price",
     xlab = "Sale Price")

hist(GrLivArea, breaks = 40, col = "lightgreen",
     main = "Distribution of GrLivArea",
     xlab = "Above Ground Living Area (sq ft)")

# =========================================
# Fit theoretical probability distributions
# =========================================

# Sale Price: Normal, Log-normal, Gamma
fit_norm_sp  <- fitdist(SalePrice, "norm")
fit_lnorm_sp <- fitdist(SalePrice, "lnorm")
fit_gamma_sp <- fitdist(SalePrice, "gamma", method = "mme", lower = c(0.001, 0.001))

# GrLivArea: Normal, Log-normal, Gamma
fit_norm_ga  <- fitdist(GrLivArea, "norm")
fit_lnorm_ga <- fitdist(GrLivArea, "lnorm")
fit_gamma_ga <- fitdist(GrLivArea, "gamma", method = "mme", lower = c(0.001, 0.001))

# =========================================
# Visual goodness-of-fit comparison (WIDE)
# =========================================

# Sale Price
options(repr.plot.width = 10, repr.plot.height = 6)

denscomp(
  list(fit_norm_sp, fit_lnorm_sp, fit_gamma_sp),
  legendtext = c("Normal", "Log-normal", "Gamma"),
  main = "Fitted Distributions: Sale Price"
)

# GrLivArea
options(repr.plot.width = 10, repr.plot.height = 6)

denscomp(
  list(fit_norm_ga, fit_lnorm_ga, fit_gamma_ga),
  legendtext = c("Normal", "Log-normal", "Gamma"),
  main = "Fitted Distributions: GrLivArea"
)


In [ ]:
# 2 HYPOTHESIS TESTING
install.packages("modeldata")
library("modeldata")
library(dplyr)
data(ames)
df <- ames
names(df)
#Auto-detect required variables
#function to detect column names
pick_col <- function(nms, patterns) {
  for (p in patterns) {
    hit <- grep(p, nms, ignore.case = TRUE)
    if (length(hit) > 0) return(nms[hit[1]])
  }
  return(NA_character_)
}

sale_col <- pick_col(names(df), c("^saleprice$", "sale[_ ]?price"))
air_col  <- pick_col(names(df), c("^centralair$", "central[_ ]?air", "air[_ ]?cond"))
hood_col <- pick_col(names(df), c("^neighbou?rhood$", "neighbou?r"))

if (is.na(sale_col) || is.na(air_col) || is.na(hood_col)) {
  stop("Could not detect SalePrice, CentralAir, or Neighborhood columns.")
}

#Rename to standard names
names(df)[names(df) == sale_col] <- "SalePrice"
names(df)[names(df) == air_col]  <- "CentralAir"
names(df)[names(df) == hood_col] <- "Neighborhood"

#Data cleaning and preparation
#Remove missing values
df <- df %>%
filter(!is.na(SalePrice), !is.na(CentralAir), !is.na(Neighborhood))

df$SalePrice <- as.numeric(df$SalePrice)

#Normalize CentralAir values
df$CentralAir <- as.character(df$CentralAir)
df$CentralAir[df$CentralAir %in% c("Y","Yes","YES","1","True","TRUE")] <- "Yes"
df$CentralAir[df$CentralAir %in% c("N","No","NO","0","False","FALSE")] <- "No"
df$CentralAir <- factor(df$CentralAir, levels = c("No", "Yes"))

df$Neighborhood <- factor(df$Neighborhood)

#Filter out Neighborhood levels with only one observation for oneway.test
#convert Neighborhood to character to allow filtering, then back to factor
df_filtered_neigh <- df %>%
  group_by(Neighborhood) %>%
  filter(n() > 1) %>%
  ungroup() %>%
  mutate(Neighborhood = factor(Neighborhood)) #Re-factor after filtering

#Hypothesis Test 1: Central Air Conditioning vs Sale Price
tapply(df$SalePrice, df$CentralAir,
       function(x) c(n = length(x),
                     mean = mean(x),
                     sd = sd(x)))

#Welch two-sample t-test
t_test_air <- t.test(SalePrice ~ CentralAir,
                     data = df,
                     var.equal = FALSE)

print(t_test_air)

#Hypothesis Test 2: Neighborhood vs Sale Price
anova_fit <- aov(SalePrice ~ Neighborhood, data = df)
summary(anova_fit)

#Post-hoc test (Tukey HSD)
TukeyHSD(anova_fit)

#Robust alternative
oneway.test(SalePrice ~ Neighborhood,
            data = df_filtered_neigh,
            var.equal = FALSE)

In [ ]:
# =========================================
# 3. REGRESSION MODELLING
# =========================================

# Load data
install.packages("modeldata")
library(modeldata)
library(ggplot2) # Added ggplot2 for density plot

data(ames)
df <- ames

# Remove missing values for selected variables
df <- df[!is.na(df$Sale_Price) &
         !is.na(df$Gr_Liv_Area) &
         !is.na(df$Overall_Cond) &
         !is.na(df$Year_Built), ]

# Ensure numeric types
df$Sale_Price   <- as.numeric(df$Sale_Price)
df$Gr_Liv_Area  <- as.numeric(df$Gr_Liv_Area)
df$Overall_Cond <- as.numeric(df$Overall_Cond)
df$Year_Built   <- as.numeric(df$Year_Built)

# Fit the multiple linear regression model
lm_model <- lm(Sale_Price ~ Gr_Liv_Area + Overall_Cond + Year_Built, data = df)
summary(lm_model)

# =========================================
# Residual diagnostics (WIDE)
# =========================================

# 1) Residuals vs Fitted (homoscedasticity)
options(repr.plot.width = 12, repr.plot.height = 6)

plot(lm_model$fitted.values,
     lm_model$residuals,
     xlab = "Fitted Values",
     ylab = "Residuals",
     main = "Residuals vs Fitted Values",
     pch = 19,
     col = "darkgray")

abline(h = 0, col = "red", lwd = 2)

# 2) Density plot of residuals (normality) - Replaced Histogram with Density Plot
options(repr.plot.width = 12, repr.plot.height = 6)

df_residuals <- data.frame(residuals = lm_model$residuals)

ggplot(df_residuals, aes(x = residuals)) +
  geom_density(fill = "lightblue", color = "blue", alpha = 0.7) +
  labs(title = "Density Plot of Residuals",
       x = "Residuals", y = "Density") +
  theme_minimal()

# 3) Q–Q plot of residuals (normality)
options(repr.plot.width = 12, repr.plot.height = 6)

qqnorm(lm_model$residuals,
       main = "Normal Q–Q Plot of Residuals")
qqline(lm_model$residuals, col = "red", lwd = 2)

In [ ]:
# =========================================
# 4.1 DESCRIPTIVE AND EXPLORATORY ANALYSIS
# =========================================

# Load Data and Required Packages
install.packages(c("modeldata", "dplyr", "ggplot2", "moments"))
library(modeldata)
library(dplyr)
library(ggplot2)
library(moments)

# Load Ames Housing dataset
data(ames)
df <- ames

# Select Key Numeric Variables
vars <- df %>%
  dplyr::select(Sale_Price, Gr_Liv_Area, Overall_Cond, Year_Built) %>%
  na.omit() %>%
  mutate(Overall_Cond = as.numeric(Overall_Cond)) %>%
  rename(
    SalePrice   = Sale_Price,
    GrLivArea   = Gr_Liv_Area,
    OverallCond = Overall_Cond,
    YearBuilt   = Year_Built
  )

# =========================================
# Summary Statistics
# =========================================

summary_stats <- data.frame(
  Variable = names(vars),
  Mean     = sapply(vars, mean),
  Median   = sapply(vars, median),
  SD       = sapply(vars, sd),
  Skewness = sapply(vars, skewness),
  Kurtosis = sapply(vars, kurtosis)
)

print(summary_stats)

# =========================================
# Wide Histograms
# =========================================

options(repr.plot.width = 14, repr.plot.height = 6)

ggplot(vars, aes(x = SalePrice)) +
  geom_histogram(bins = 40, fill = "steelblue", color = "black") + # Changed color to black
  labs(title = "Distribution of Sale Price",
       x = "Sale Price", y = "Count") +
  theme_minimal()

ggplot(vars, aes(x = GrLivArea)) +
  geom_histogram(bins = 40, fill = "darkgreen", color = "black") + # Changed color to black
  labs(title = "Distribution of Above-Ground Living Area",
       x = "GrLivArea (sq ft)", y = "Count") +
  theme_minimal()

# =========================================
# Wide Box Plots
# =========================================

options(repr.plot.width = 14, repr.plot.height = 6)

ggplot(vars, aes(y = SalePrice)) +
  geom_boxplot(fill = "orange") +
  labs(title = "Box Plot of Sale Price",
       y = "Sale Price") +
  theme_minimal()

ggplot(vars, aes(y = GrLivArea)) +
  geom_boxplot(fill = "purple") +
  labs(title = "Box Plot of GrLivArea",
       y = "GrLivArea (sq ft)") +
  theme_minimal()

# =========================================
# Wide Scatter Plots
# =========================================

options(repr.plot.width = 14, repr.plot.height = 6)

ggplot(vars, aes(x = GrLivArea, y = SalePrice)) +
  geom_point(alpha = 0.4) +
  geom_smooth(method = "lm", se = FALSE, color = "red") +
  labs(title = "Sale Price vs Living Area",
       x = "GrLivArea (sq ft)", y = "Sale Price") +
  theme_minimal()

ggplot(vars, aes(x = OverallCond, y = SalePrice)) +
  geom_point(alpha = 0.4) +
  geom_smooth(method = "lm", se = FALSE, color = "blue") +
  labs(title = "Sale Price vs Overall Condition",
       x = "Overall Condition", y = "Sale Price") +
  theme_minimal()


In [ ]:
# =============================
# 4.2 DISTRIBUTION CONSTRUCTION
# =============================

# ---- Load required packages ----
install.packages(c("fitdistrplus", "modeldata", "ggplot2"))
library(fitdistrplus)
library(ggplot2)
library(modeldata)

# ---- Load dataset ----
data(ames)
df <- ames

# Rename columns for convenience
names(df)[names(df) == "Sale_Price"]  <- "SalePrice"
names(df)[names(df) == "Gr_Liv_Area"] <- "GrLivArea"

# ---- Extract variables and remove missing values ----
SalePrice <- as.numeric(df$SalePrice)
GrLivArea <- as.numeric(df$GrLivArea)

SalePrice <- SalePrice[!is.na(SalePrice)]
GrLivArea <- GrLivArea[!is.na(GrLivArea)]

# ======================================================
# 1) Empirical distributions (histograms + KDE)
# ======================================================
options(repr.plot.width = 14, repr.plot.height = 6)
par(mfrow = c(1,2))

hist(SalePrice, prob = TRUE, breaks = 40,
     col = "lightblue",
     main = "Empirical Distribution: SalePrice",
     xlab = "Sale Price")
lines(density(SalePrice), col = "blue", lwd = 2)

hist(GrLivArea, prob = TRUE, breaks = 40,
     col = "lightgreen",
     main = "Empirical Distribution: GrLivArea",
     xlab = "Above-Ground Living Area")
lines(density(GrLivArea), col = "darkgreen", lwd = 2)


# ======================================================
# 2) Fit theoretical distributions
# ======================================================

# ---- SalePrice ----
fit_norm_sp  <- fitdist(SalePrice, "norm")
fit_lnorm_sp <- fitdist(SalePrice, "lnorm")
fit_gamma_sp <- fitdist(SalePrice, "gamma",
                         method = "mme",
                         lower = c(0.001, 0.001))

# ---- GrLivArea ----
fit_norm_ga  <- fitdist(GrLivArea, "norm")
fit_lnorm_ga <- fitdist(GrLivArea, "lnorm")
fit_gamma_ga <- fitdist(GrLivArea, "gamma",
                         method = "mme",
                         lower = c(0.001, 0.001))


# ======================================================
# 3) Density comparison: empirical vs theoretical
# ======================================================

# ---- SalePrice ----
options(repr.plot.width = 10, repr.plot.height = 6)
denscomp(
  list(fit_norm_sp, fit_lnorm_sp, fit_gamma_sp),
  legendtext = c("Normal", "Log-normal", "Gamma"),
  main = "SalePrice: Empirical vs Theoretical Fits"
)

# ---- GrLivArea ----
options(repr.plot.width = 10, repr.plot.height = 6)
denscomp(
  list(fit_norm_ga, fit_lnorm_ga, fit_gamma_ga),
  legendtext = c("Normal", "Log-normal", "Gamma"),
  main = "GrLivArea: Empirical vs Theoretical Fits"
)


# ======================================================
# 4) Q–Q plots (distributional adequacy)
# ======================================================

# ---- SalePrice ----
options(repr.plot.width = 10, repr.plot.height = 6)
qqcomp(
  list(fit_norm_sp, fit_lnorm_sp, fit_gamma_sp),
  legendtext = c("Normal", "Log-normal", "Gamma"),
  main = "Q–Q Plots: SalePrice"
)

# ---- GrLivArea ----
options(repr.plot.width = 10, repr.plot.height = 6)
qqcomp(
  list(fit_norm_ga, fit_lnorm_ga, fit_gamma_ga),
  legendtext = c("Normal", "Log-normal", "Gamma"),
  main = "Q–Q Plots: GrLivArea"
)


# ======================================================
# 5) Statistical goodness-of-fit comparison
# ======================================================

# ---- SalePrice ----
gof_sp <- gofstat(list(fit_norm_sp, fit_lnorm_sp, fit_gamma_sp))
print(gof_sp)

# ---- GrLivArea ----
gof_ga <- gofstat(list(fit_norm_ga, fit_lnorm_ga, fit_gamma_ga))
print(gof_ga)


# ======================================================
# 6) Log-transformation confirmation
# ======================================================

logSalePrice <- log(SalePrice)
logGrLivArea <- log(GrLivArea)

options(repr.plot.width = 14, repr.plot.height = 6)
par(mfrow = c(1,2))

qqnorm(logSalePrice, main = "Q–Q Plot: log(SalePrice)")
qqline(logSalePrice, col = "red")

qqnorm(logGrLivArea, main = "Q–Q Plot: log(GrLivArea)")
qqline(logGrLivArea, col = "red")


In [ ]:
# =========================================================
# 4.3 HYPOTHESIS TESTING
# =========================================================

# Load required packages (install once if needed)
# install.packages(c("modeldata", "dplyr"))
library(modeldata)
library(dplyr)

library(ggplot2)

# Load dataset
data(ames)
df <- ames

# =========================================================
# 1. AUTO-DETECT & STANDARDIZE VARIABLE NAMES
# =========================================================

pick_col <- function(nms, patterns) {
  for (p in patterns) {
    hit <- grep(p, nms, ignore.case = TRUE)
    if (length(hit) > 0) return(nms[hit[1]])
  }
  NA_character_
}

sale_col <- pick_col(names(df), c("^saleprice$", "sale[_ ]?price"))
air_col  <- pick_col(names(df), c("^centralair$", "central[_ ]?air"))
hood_col <- pick_col(names(df), c("^neighbou?rhood$", "neighbou?r"))

if (is.na(sale_col) || is.na(air_col) || is.na(hood_col)) {
  stop("Required variables could not be detected.")
}

names(df)[names(df) == sale_col] <- "SalePrice"
names(df)[names(df) == air_col]  <- "CentralAir"
names(df)[names(df) == hood_col] <- "Neighborhood"

# =========================================================
# 2. DATA CLEANING & PREPARATION
# =========================================================

df <- df %>%
  filter(!is.na(SalePrice),
         !is.na(CentralAir),
         !is.na(Neighborhood))

df$SalePrice <- as.numeric(df$SalePrice)

# Normalize CentralAir values
df$CentralAir <- as.character(df$CentralAir)
df$CentralAir[df$CentralAir %in% c("Y","Yes","YES","1","True","TRUE")] <- "Yes"
df$CentralAir[df$CentralAir %in% c("N","No","NO","0","False","FALSE")] <- "No"
df$CentralAir <- factor(df$CentralAir, levels = c("No", "Yes"))

df$Neighborhood <- factor(df$Neighborhood)

# Remove neighborhoods with only one observation (needed for Welch ANOVA)
df_neigh <- df %>%
  group_by(Neighborhood) %>%
  filter(n() > 1) %>%
  ungroup()

# =========================================================
# 3. TWO-SAMPLE T-TEST
# Central Air Conditioning vs Sale Price
# =========================================================

tapply(df$SalePrice, df$CentralAir,
       function(x) c(n = length(x),
                     mean = mean(x),
                     sd = sd(x)))

t_test_air <- t.test(SalePrice ~ CentralAir,
                     data = df,
                     var.equal = FALSE)
print(t_test_air)
# =========================================================
# GRAPH: Two-Sample t-test (Central Air vs Sale Price)
# =========================================================


ggplot(df, aes(x = CentralAir, y = SalePrice, fill = CentralAir)) +
  geom_boxplot(alpha = 0.7, outlier.shape = NA) +
  geom_jitter(width = 0.15, alpha = 0.25, size = 1) +
  labs(
    title = "Sale Price by Central Air Conditioning",
    x = "Central Air Conditioning",
    y = "Sale Price"
  ) +
  theme_minimal() +
  theme(legend.position = "none")


# =========================================================
# 4. ONE-WAY ANOVA
# Neighborhood vs Sale Price
# =========================================================

anova_fit <- aov(SalePrice ~ Neighborhood, data = df)
summary(anova_fit)

# Post-hoc comparison
TukeyHSD(anova_fit)

# =========================================================
# 5. WELCH ANOVA (ROBUST ALTERNATIVE)
# =========================================================

oneway.test(SalePrice ~ Neighborhood,
            data = df_neigh,
            var.equal = FALSE)

# =========================================================
# 6. CHI-SQUARE TEST (χ²)
# Neighborhood vs Central Air Conditioning
# =========================================================

air_neigh_table <- table(df$Neighborhood, df$CentralAir)
chisq.test(air_neigh_table)


In [ ]:
# =========================================
# 4.4 REGRESSION ANALYSIS
# =========================================

# Load data
library(modeldata)
data(ames)
df <- ames

# Standardize column names
names(df)[names(df) == "Sale_Price"]   <- "SalePrice"
names(df)[names(df) == "Gr_Liv_Area"]  <- "GrLivArea"
names(df)[names(df) == "Overall_Cond"] <- "OverallQual"
names(df)[names(df) == "Year_Built"]   <- "YearBuilt"

# Remove missing values
df <- df[!is.na(df$SalePrice) &
         !is.na(df$GrLivArea) &
         !is.na(df$OverallQual) &
         !is.na(df$YearBuilt), ]

# Ensure correct data types
df$SalePrice   <- as.numeric(df$SalePrice)
df$GrLivArea   <- as.numeric(df$GrLivArea)
df$OverallQual <- as.numeric(df$OverallQual)
df$YearBuilt   <- as.numeric(df$YearBuilt)

# =========================================
# Train–Test Split (Out-of-Sample Evaluation)
# =========================================

set.seed(123)

n <- nrow(df)
train_idx <- sample(seq_len(n), size = 0.7 * n)

train_data <- df[train_idx, ]
test_data  <- df[-train_idx, ]

# =========================================
# Fit the Multiple Linear Regression Model
# =========================================

lm_model <- lm(SalePrice ~ GrLivArea + OverallQual + YearBuilt,
               data = train_data)

summary(lm_model)

# =========================================
# In-Sample Diagnostics (WIDE)
# =========================================

# 1) Residuals vs Fitted
options(repr.plot.width = 12, repr.plot.height = 6)

plot(lm_model$fitted.values,
     lm_model$residuals,
     xlab = "Fitted Values",
     ylab = "Residuals",
     main = "Residuals vs Fitted",
     pch = 19,
     col = "darkgray")

abline(h = 0, col = "red", lwd = 2)

# 2) Normal Q–Q Plot
options(repr.plot.width = 12, repr.plot.height = 6)

qqnorm(lm_model$residuals,
       main = "Normal Q–Q Plot of Residuals")
qqline(lm_model$residuals, col = "red", lwd = 2)

# 3) Scale–Location Plot
options(repr.plot.width = 12, repr.plot.height = 6)

plot(lm_model$fitted.values,
     sqrt(abs(lm_model$residuals)),
     xlab = "Fitted Values",
     ylab = "Sqrt(|Residuals|)",
     main = "Scale–Location Plot",
     pch = 19,
     col = "darkgray")

# 4) Residuals vs Leverage (Cook’s Distance)
options(repr.plot.width = 12, repr.plot.height = 6)

plot(lm_model, which = 5)

# =========================================
# R² and Adjusted R²
# =========================================

summary(lm_model)$r.squared
summary(lm_model)$adj.r.squared

# =========================================
# Out-of-Sample Prediction and RMSE
# =========================================

# Predict Sale Prices on Test Data
test_pred <- predict(lm_model, newdata = test_data)

# Compute RMSE
rmse <- sqrt(mean((test_data$SalePrice - test_pred)^2))
rmse

# =========================================
# Example Prediction for a New House
# =========================================

new_house <- data.frame(
  GrLivArea   = 2000,
  OverallQual = 7,
  YearBuilt   = 2005
)

predict(lm_model, newdata = new_house)


In [ ]:
# =========================================
# 5. ADDITIONAL — Extended Analysis
# Model Improvements and Advanced Exploration
# =========================================

# =========================================
# Log-Transformation of Sale Price
# =========================================

# Log-transform SalePrice
train_data$logSalePrice <- log(train_data$SalePrice)
test_data$logSalePrice  <- log(test_data$SalePrice)

log_lm <- lm(logSalePrice ~ GrLivArea + OverallQual + YearBuilt,
             data = train_data)

summary(log_lm)

# =========================================
# Diagnostics for Log-Linear Model (WIDE)
# =========================================

# 1) Residuals vs Fitted
options(repr.plot.width = 12, repr.plot.height = 6)

plot(log_lm$fitted.values,
     log_lm$residuals,
     xlab = "Fitted Values",
     ylab = "Residuals",
     main = "Log-Linear: Residuals vs Fitted",
     pch = 19,
     col = "darkgray")

abline(h = 0, col = "red", lwd = 2)

# 2) Normal Q–Q Plot
options(repr.plot.width = 12, repr.plot.height = 6)

qqnorm(log_lm$residuals,
       main = "Log-Linear: Normal Q–Q Plot")
qqline(log_lm$residuals, col = "red", lwd = 2)

# 3) Scale–Location Plot
options(repr.plot.width = 12, repr.plot.height = 6)

plot(log_lm$fitted.values,
     sqrt(abs(log_lm$residuals)),
     xlab = "Fitted Values",
     ylab = "Sqrt(|Residuals|)",
     main = "Log-Linear: Scale–Location Plot",
     pch = 19,
     col = "darkgray")

# 4) Residuals vs Leverage (Cook’s Distance)
options(repr.plot.width = 12, repr.plot.height = 6)

plot(log_lm, which = 5)

# =========================================
# Interaction Effects Between Features
# =========================================

interaction_lm <- lm(logSalePrice ~ GrLivArea * OverallQual + YearBuilt,
                     data = train_data)

summary(interaction_lm)

# =========================================
# Polynomial Regression (Non-Linear Effect of Size)
# =========================================

poly_lm <- lm(logSalePrice ~ poly(GrLivArea, 2) + OverallQual + YearBuilt,
              data = train_data)

summary(poly_lm)

# =========================================
# Model Comparison Using Out-of-Sample RMSE
# =========================================

rmse_fun <- function(actual, predicted) {
  sqrt(mean((actual - predicted)^2))
}

# Back-transform predictions
pred_base <- exp(predict(lm_model, test_data))
pred_log  <- exp(predict(log_lm, test_data))
pred_int  <- exp(predict(interaction_lm, test_data))
pred_poly <- exp(predict(poly_lm, test_data))

rmse_results <- data.frame(
  Model = c("Linear", "Log-Linear", "Interaction", "Polynomial"),
  RMSE = c(
    rmse_fun(test_data$SalePrice, pred_base),
    rmse_fun(test_data$SalePrice, pred_log),
    rmse_fun(test_data$SalePrice, pred_int),
    rmse_fun(test_data$SalePrice, pred_poly)
  )
)

rmse_results


In [ ]:
# =========================================
# Hypothesis Test 3: Chi-Square Test
# CentralAir vs Neighborhood
# =========================================

chisq_table <- table(df$CentralAir, df$Neighborhood)
chisq_test <- chisq.test(chisq_table)

chisq_test


In [ ]:
v